In [ ]:
# Import required packages
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Directories
CODE_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Code")
DATA_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Data")

# File names
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

# Parallel processing
N_JOBS = 16  # Number of parallel jobs for OOS predictions

# LASSO parameters
MAX_ITER = 10000  # Increased iterations for convergence
CV_FOLDS = 5  # Number of cross-validation folds

In [ ]:
data = pd.read_pickle(INPUT_DATA)

In [ ]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Select features
FEATURES = ['net_sentiment', 'log_volume']

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

In [ ]:
# Add date column and sort
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])
model_data = model_data.sort_values('date')
model_data['year_month'] = model_data['date'].dt.to_period('M')

# In-Sample LASSO regression

In [ ]:
# In-sample training window (adjustable)
TRAIN_START_DATE = '2011-01-01'
TRAIN_END_DATE = '2011-12-31'

# Filter data to training window
train_start = pd.to_datetime(TRAIN_START_DATE)
train_end = pd.to_datetime(TRAIN_END_DATE)
train_mask = (model_data['date'] >= train_start) & (model_data['date'] <= train_end)

X_train = model_data.loc[train_mask, FEATURES]
y_train = model_data.loc[train_mask, TARGET]

print(f"Training window: {TRAIN_START_DATE} to {TRAIN_END_DATE}")
print(f"Training samples: {len(X_train):,}")

# Normalize features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Fit LASSO regression model with cross-validation for optimal alpha
import time
start_time = time.time()

lasso_model = LassoCV(cv=CV_FOLDS, random_state=42, n_jobs=-1, max_iter=MAX_ITER)
lasso_model.fit(X_train_scaled, y_train)

elapsed_time = time.time() - start_time

# Make predictions
y_pred = lasso_model.predict(X_train_scaled)

# Evaluate performance
r2 = r2_score(y_train, y_pred)
mse = mean_squared_error(y_train, y_pred)
rmse = np.sqrt(mse)

print(f"\nIn-Sample LASSO Regression Results")
print("=" * 50)
print(f"Optimal alpha (regularization): {lasso_model.alpha_:.6f}")
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print("\nCoefficients (on standardized features):")
for feature, coef in zip(FEATURES, lasso_model.coef_):
    print(f"  {feature}: {coef:.6f}")
print(f"  Intercept: {lasso_model.intercept_:.6f}")
print(f"\nTraining time: {elapsed_time:.2f} seconds")

# OOS predictions

In [ ]:
# Out-of-sample predictions with MONTHLY TRAINING but DAILY PREDICTIONS
# The model is trained once per month (at month-end) and used to predict all days in the following month

# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW = 252  # Rolling window size in trading days (252 = one year)

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Rolling window: {WINDOW} trading days")
print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")

def train_and_predict_month(month_idx, pred_month, model_data, unique_dates, oos_dates,
                            FEATURES, TARGET, WINDOW, CV_FOLDS, MAX_ITER):
    """Train LASSO model for a single month and generate predictions for all days in that month."""
    predictions = []
    alpha_info = None
    
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    
    if len(month_dates) == 0:
        return predictions, alpha_info
    
    # Training cutoff: end of the previous month (first day of pred_month - 1 day)
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    
    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return predictions, alpha_info
    last_train_date = train_dates[-1]
    
    # ROLLING WINDOW: Train on last WINDOW trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return predictions, alpha_info
    
    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    train_mask = (model_data['date'] >= start_date) & (model_data['date'] <= last_train_date)
    X_train = model_data.loc[train_mask, FEATURES]
    y_train = model_data.loc[train_mask, TARGET]
    
    if len(X_train) == 0:
        return predictions, alpha_info
    
    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Fit LASSO with CV (use n_jobs=1 inside since we're parallelizing across months)
    lasso_model = LassoCV(cv=CV_FOLDS, random_state=42, n_jobs=1, max_iter=MAX_ITER)
    lasso_model.fit(X_train_scaled, y_train)
    alpha_info = {'month': pred_month, 'alpha': lasso_model.alpha_}
    
    # Use this model to predict for all days in the month
    for pred_date in month_dates:
        test_mask = model_data['date'] == pred_date
        X_test = model_data.loc[test_mask, FEATURES]
        
        if len(X_test) == 0:
            continue
        
        # Transform test features using the same scaler
        X_test_scaled = scaler.transform(X_test)
        
        test_indices = model_data.index[test_mask]
        y_pred = lasso_model.predict(X_test_scaled)
        
        for idx, pred in zip(test_indices, y_pred):
            predictions.append({
                'date': pred_date,
                'index': idx,
                'prediction': pred
            })
    
    return predictions, alpha_info

# Execute in parallel
print(f"\nRunning parallel processing with {N_JOBS} jobs...")
def get_month_slice(pred_month):
    """Pre-slice model_data to the rows needed for this month's rolling window + predictions,
    so each parallel worker receives a small DataFrame instead of a full copy of model_data."""
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    if len(month_dates) == 0:
        return model_data.iloc[0:0]
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return model_data.iloc[0:0]
    last_train_date = train_dates[-1]
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return model_data.iloc[0:0]
    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    end_date = month_dates.max()
    return model_data.loc[(model_data['date'] >= start_date) & (model_data['date'] <= end_date)]

all_results = Parallel(n_jobs=N_JOBS, verbose=10, backend='threading')(
    delayed(train_and_predict_month)(
        month_idx, pred_month, get_month_slice(pred_month), unique_dates, oos_dates,
        FEATURES, TARGET, WINDOW, CV_FOLDS, MAX_ITER
    )
    for month_idx, pred_month in enumerate(oos_months)
)

# Collect results
predictions = []
alphas = []
for month_predictions, alpha_info in all_results:
    predictions.extend(month_predictions)
    if alpha_info is not None:
        alphas.append(alpha_info)

print(f"\nCompleted.")
print(f"Total predictions: {len(predictions):,}")

In [ ]:
# Summary of selected alpha values
print("Selected Alpha (Regularization Parameter) Summary")
print("=" * 50)

if alphas:
    alphas_df = pd.DataFrame(alphas)
    print(f"\nRolling {WINDOW}-day Window:")
    print(f"  Mean alpha: {alphas_df['alpha'].mean():.6f}")
    print(f"  Std alpha: {alphas_df['alpha'].std():.6f}")
    print(f"  Min alpha: {alphas_df['alpha'].min():.6f}")
    print(f"  Max alpha: {alphas_df['alpha'].max():.6f}")

In [ ]:
# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"Non-null predictions: {predictions_df['prediction'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

In [ ]:
# Save predictions to Data directory
OUTPUT_FILE = MODEL_DATA_DIR / f"predictions_lasso_input={len(FEATURES)}.pkl"
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")